# Setup

In [ ]:
#importing necessary libraries
import re
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# Make sure the notebook is always run from the project root
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Working directory:", Path.cwd())

# global plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Folder paths
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1.Data Loading

In [ ]:
def _resolve_path(raw_dir: Path, filename: str) -> Path:
    """Return whichever of filename / filename+'.gz' exists in raw_dir."""
    plain = raw_dir / filename
    gz = raw_dir / f'{filename}.gz'
    if plain.exists():
        return plain
    if gz.exists():
        return gz
    raise FileNotFoundError(
        f'Could not find {plain} or {gz}. ')

In [ ]:
def load_listings(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Load the detailed listings dataset (one row per Airbnb listing)."""
    path = _resolve_path(raw_dir, 'listings.csv')
    compression = 'gzip' if path.suffix == '.gz' else None
    return pd.read_csv(path, compression=compression, low_memory=False)

In [ ]:
def load_reviews(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Load the detailed reviews dataset (one row per review)."""
    path = _resolve_path(raw_dir, 'reviews.csv')
    compression = 'gzip' if path.suffix == '.gz' else None
    return pd.read_csv(path, compression=compression, low_memory=False)

In [ ]:
def load_calendar(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Load the calendar dataset (one row per listing per date)."""
    path = _resolve_path(raw_dir, 'calendar.csv')
    compression = 'gzip' if path.suffix == '.gz' else None
    return pd.read_csv(path, compression=compression, low_memory=False)


In [ ]:
def load_neighbourhoods(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Load the neighbourhood -> borough lookup table."""
    path = _resolve_path(raw_dir, 'neighbourhoods.csv')
    return pd.read_csv(path)

In [ ]:
def load_neighbourhoods_geo(raw_dir: Path = RAW_DIR):
    """Load neighbourhood boundary polygons as a GeoDataFrame."""
    path = raw_dir / 'neighbourhoods.geojson'
    if not path.exists():
        raise FileNotFoundError(f'Could not find {path}.')
    return gpd.read_file(path)

In [ ]:
#Loading all 5 raw datasets
df_listings_raw = load_listings()
df_reviews = load_reviews()
df_calender = load_calendar()
df_nhood = load_neighbourhoods()
neighbourhood_geo = load_neighbourhoods_geo()

#Shape check
print(f'Listings shape:        {df_listings_raw.shape}')
print(f'Reviews shape:         {df_reviews.shape}')
print(f'Calendar shape:        {df_calender.shape}')
print(f'Neighbourhoods shape:  {df_nhood.shape}')
print(f'Neighbourhoods (geo):  {neighbourhood_geo.shape}')

In [ ]:
#Feature names of the listings dataset
print(f'\nListings columns ({len(df_listings_raw.columns)}):')
print(list(df_listings_raw.columns))

In [ ]:
df_listings_raw.head(3)

In [ ]:
print(df_listings_raw['price'].dtype)
print(df_listings_raw['price'].head())

# 2.Missing Values

In [ ]:
# Compute the fraction of missing values per column, keep only columns with >1% missing
missing_raw = df_listings_raw.isnull().mean().sort_values(ascending=False)
missing_raw_top = missing_raw[missing_raw > 0.01].head(25)

In [ ]:
# Highlighting the price column in a different color, since it's our target variable
colors = ['teal' if col == 'price' else 'salmon' for col in missing_raw_top.index]

plt.figure(figsize=(10, 7))

missing_raw_top.plot(kind='barh', color=colors)

plt.title('Missing Value Rate — Raw Data (columns > 1% missing)')
plt.xlabel('Fraction Missing')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
print('Columns with >50% missing:')
print(list(missing_raw[missing_raw > 0.5].index))

print(f'\nprice missing: {df_listings_raw["price"].isna().mean():.1%}')

# 3.Data Cleaning

In [ ]:
def clean_price_column(df: pd.DataFrame, price_col: str = 'price') -> pd.DataFrame:
    """Convert price from a string to a float. Keeps the original string in `price_raw` for traceability."""
    df = df.copy()
    df[f'{price_col}_raw'] = df[price_col]
    df[price_col] = (df[price_col].astype(str).str.replace(r'[\$,]', '', regex=True).replace('nan', np.nan).astype(float))
    
    return df

In [ ]:
def remove_price_outliers(df: pd.DataFrame, price_col: str = 'price', lower_bound: float = 0, upper_quantile: float = 0.99,) -> pd.DataFrame:
    """Drop listings with missing price, price <= lower_bound, or price above the upper quantile."""
    df = df.copy()
    upper_bound = df[price_col].quantile(upper_quantile)

    mask = ((df[price_col] > lower_bound) & 
            (df[price_col] <= upper_bound) & 
            df[price_col].notna())

    n_removed = len(df) - mask.sum()
    print(f'remove_price_outliers: removed {n_removed} listings '
          f'({n_removed / len(df):.1%}) — missing price, price<=0, or price > '
          f'€{upper_bound:.0f} (top {1 - upper_quantile:.0%})')

    return df[mask].reset_index(drop=True)

In [ ]:
def parse_bathrooms(df: pd.DataFrame) -> pd.DataFrame:
    """Create a clean numeric `bathrooms_parsed` column, preferring the numeric `bathrooms` field and falling back to parsing `bathrooms_text` when `bathrooms` is missing."""
    df = df.copy()

    def _parse_row(row):
        if pd.notna(row.get('bathrooms')) and row['bathrooms'] > 0:
            return float(row['bathrooms'])
        
        text = str(row.get('bathrooms_text', ''))
        match = re.search(r'([\d\.]+)', text)
        
        return float(match.group(1)) if match else np.nan

    df['bathrooms_parsed'] = df.apply(_parse_row, axis=1)
    return df

In [ ]:
def parse_date_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert date-as-string columns to proper datetime dtype. host_since, first_review, and last_review arrive as ISO date strings.

    Missing values are legitimate here, not data errors:
    - host_since missing -> a handful of incomplete host records
    - first_review / last_review missing -> the listing has zero reviews

    A `_was_missing` flag is added for each column so this information isn't silently lost.
    """
    df = df.copy()
    
    date_cols = ['host_since', 'first_review', 'last_review']
    
    for col in date_cols:
        if col in df.columns:
            df[f'{col}_was_missing'] = df[col].isna().astype(int)
            df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

In [ ]:
def parse_percentage_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert percentage-as-string columns to floats."""
    df = df.copy()

    def _parse_pct(value):
        try:
            return float(str(value).replace('%', '')) / 100
        except (ValueError, TypeError):
            return np.nan

    for col in ['host_response_rate', 'host_acceptance_rate']:
        if col in df.columns:
            df[col] = df[col].apply(_parse_pct)
    return df

In [ ]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing values for common modeling columns:
    - percentage columns parsed first
    - numeric columns -> median fill + `_was_missing` flag
    - bedrooms/beds -> fill with 1
    - categoricals -> fill with \"Unknown\"
    """
    df = df.copy()
    df = parse_percentage_columns(df)

    review_score_cols = [c for c in df.columns if c.startswith('review_scores_')]
    numeric_fill_median = review_score_cols + [c for c in ['host_response_rate', 'host_acceptance_rate','reviews_per_month', 'bathrooms_parsed'] if c in df.columns]

    for col in numeric_fill_median:
        df[f'{col}_was_missing'] = df[col].isna().astype(int)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

    for col in ['bedrooms', 'beds']:
        if col in df.columns:
            df[f'{col}_was_missing'] = df[col].isna().astype(int)
            df[col] = df[col].fillna(1)
        
    categorical_cols = [c for c in ['neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                                        'property_type', 'room_type',
                                        'host_response_time', 'host_is_superhost', 'has_availability',] if c in df.columns]
    
    for col in categorical_cols:
        df[col] = df[col].fillna('Unknown')

    return df

In [ ]:
def clean_listings(df: pd.DataFrame) -> pd.DataFrame:
    """Run the full cleaning pipeline on raw listings data."""
    df = clean_price_column(df)
    df = remove_price_outliers(df)
    df = parse_bathrooms(df)
    df = parse_date_columns(df)
    df = handle_missing_values(df)
    return df

In [ ]:
df_listings = clean_listings(df_listings_raw)

# Compare shape and price stats before vs. after cleaning
print(f'\nShape before: {df_listings_raw.shape}')
print(f'Shape after:  {df_listings.shape}')

print(f'\nPrice dtype: {df_listings["price"].dtype}')

print(f'Price range: €{df_listings["price"].min():.0f} – €{df_listings["price"].max():.0f}')
print(f'Price median: €{df_listings["price"].median():.0f}')

In [ ]:
# Sanity check: any missing values left in key modeling columns?
key_cols = [
    'price', 'accommodates', 'bedrooms', 'beds', 'bathrooms_parsed',
    'room_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'latitude', 'longitude',
]
key_cols = [c for c in key_cols if c in df_listings.columns]

remaining = df_listings[key_cols].isnull().sum()
print('Missing values in key columns after cleaning:')
print(remaining[remaining > 0] if remaining.sum() > 0 else 'None')

# 4.Exploratory Data Analysis

### 4.1 Price Distribution

In [ ]:
# Plot price distribution both on the raw scale and on a log scale, side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_listings['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price  (€/night)')
axes[0].set_ylabel('Count')

#because the price distribution is right skewed we also check the log-transformed distribution (recommended the use log scale for modeling)
axes[1].hist(np.log1p(df_listings['price']), bins=60, color='coral', edgecolor='white')
axes[1].set_title('Log-Price Distribution')
axes[1].set_xlabel('log(1 + Price)')

plt.tight_layout()
plt.show()

print(df_listings['price'].describe())

### 4.2 Price by Room Type

In [ ]:
# Boxplot of price per room type, ordered from most to least expensive (median)
plt.figure(figsize=(8, 5))

order = df_listings.groupby('room_type')['price'].median().sort_values(ascending=False).index

sns.boxplot(data=df_listings, x='room_type', y='price', order=order, showfliers=False, color='firebrick')

plt.title('Price by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Price (€/night)')
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

print(df_listings.groupby('room_type')['price'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False))

### 4.3 Price by Neighbourhood

In [ ]:
# Two side-by-side views of spatial price differences: by borough (coarse) and
# by individual neighbourhood (fine-grained, top 15 only to keep it readable)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

borough_order = (df_listings.groupby('neighbourhood_group_cleansed')['price'].median().sort_values())

borough_order.plot(kind='barh', ax=axes[0], color='turquoise')
axes[0].set_title('Median Price by Borough (neighbourhood_group_cleansed)')
axes[0].set_xlabel('Median Price (€/night)')

top_hoods = (df_listings.groupby('neighbourhood_cleansed')['price'].median().sort_values(ascending=False).head(15).sort_values())

top_hoods.plot(kind='barh', ax=axes[1], color='mediumorchid')
axes[1].set_title('Median Price by Neighbourhood (Top 15)')
axes[1].set_xlabel('Median Price (€/night)')

plt.tight_layout()
plt.show()

### 4.4 Price vs Accommodates

In [ ]:
# Show the price-accommodates relationship
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=df_listings, x='accommodates', y='price', ax=axes[0], showfliers=False, color='mediumseagreen')

axes[0].set_title('Price vs. Accommodates (boxplot)')
axes[0].set_xlabel('Accommodates')
axes[0].set_ylabel('Price (€/night)')

axes[1].scatter(df_listings['accommodates'], df_listings['price'], alpha=0.5, s=10, color='lightseagreen')
axes[1].set_title('Price vs. Accommodates (scatter)')
axes[1].set_xlabel('Accommodates')
axes[1].set_ylabel('Price (€/night)')

plt.tight_layout()
plt.show()

# Quantify the relationship with a correlation coefficient
print('Correlation (accommodates, price):', df_listings['accommodates'].corr(df_listings['price']).round(3))

### 4.5 Spatial Overview

In [ ]:
# Scatter plot of every listing on the map, coloured by log-price, with Berlin's
# neighbourhood boundaries drawn underneath for geographic context
fig, ax = plt.subplots(figsize=(8, 8))

neighbourhood_geo.boundary.plot(ax=ax, color='rosybrown', linewidth=0.7, zorder=1)

sc = ax.scatter(df_listings['longitude'], df_listings['latitude'],
                c=np.log1p(df_listings['price']), cmap='magma',
                alpha=0.7, s=5, zorder=2)

plt.colorbar(sc, label='log(1 + price)', ax=ax, shrink=0.7, fraction=0.046, pad=0.04)

ax.set_title('Berlin Airbnb Listings — Price Heatmap (scatter)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

### 4.6 Choropleth Map (Neighbourhood Boundaries)

In [ ]:
# Cross-check: do neighbourhood names in listings match the geojson exactly?
listings_hoods = set(df_listings['neighbourhood_cleansed'].dropna().unique())
geo_hoods = set(neighbourhood_geo['neighbourhood'].dropna().unique())


print(f'Neighbourhoods in listings: {len(listings_hoods)}')
print(f'Neighbourhoods in geojson:  {len(geo_hoods)}')
print(f'In listings but missing from geojson: {listings_hoods - geo_hoods or "none "}')

In [ ]:
# Compute median price and listing count per neighbourhood
median_price_by_hood = (df_listings.groupby('neighbourhood_cleansed')['price'].median().reset_index().rename(columns={'neighbourhood_cleansed': 'neighbourhood', 'price': 'median_price'}))

listing_count_by_hood = (df_listings.groupby('neighbourhood_cleansed').size().reset_index(name='n_listings').rename(columns={'neighbourhood_cleansed': 'neighbourhood'}))

hoods_map = (neighbourhood_geo.merge(median_price_by_hood, on='neighbourhood', how='left').merge(listing_count_by_hood, on='neighbourhood', how='left'))

# Check how many neighbourhoods ended up with no price data after the merge
print(f'Neighbourhoods with no price data after cleaning: '
      f'{hoods_map["median_price"].isna().sum()} / {len(hoods_map)}')

In [ ]:
# Two choropleth maps side by side: median price and listing density per neighbourhood
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

hoods_map.plot(column='median_price', cmap='RdYlGn_r', legend=True, ax=axes[0],
               edgecolor='white', linewidth=0.3,
               missing_kwds={'color': 'lightgrey', 'label': 'No data'},
               legend_kwds={'label': 'Median Price (€/night)', 'shrink': 0.7})

axes[0].set_title('Median Price by Neighbourhood')
axes[0].set_axis_off()

hoods_map.plot(column='n_listings', cmap='Blues', legend=True, ax=axes[1],
               edgecolor='white', linewidth=0.3,
               missing_kwds={'color': 'lightgrey', 'label': 'No data'},
               legend_kwds={'label': 'Number of Listings', 'shrink': 0.7})

axes[1].set_title('Listing Density by Neighbourhood')
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

print('Most expensive neighbourhoods (median price):')
print(hoods_map.nlargest(5, 'median_price')[['neighbourhood', 'median_price', 'n_listings']].to_string(index=False))

### 4.6 Missing Values (After Cleaning)

In [ ]:
# Recompute missing-value rates on the cleaned dataframe, to see what's left
missing_clean = df_listings.isnull().mean().sort_values(ascending=False)
missing_clean_top = missing_clean[missing_clean > 0]

In [ ]:
# Plot remaining missing values (if any), otherwise confirm the dataset is fully clean
if len(missing_clean_top) > 0:
    
    plt.figure(figsize=(10, max(3, 0.3 * len(missing_clean_top))))
    
    missing_clean_top.plot(kind='barh', color='salmon')
    
    plt.title('Remaining Missing Values — After Cleaning')
    plt.xlabel('Fraction Missing')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

else:
    print('No missing values remain.')

print(missing_clean_top)

**Why are there still missing values here?** 

The columns above fall into a few different buckets — not all missingness needs the same treatment, and none of it blocks Part 2/3 from using this dataset:

1. **Unused / out-of-scope columns** — `calendar_updated` (100% empty in this Berlin snapshot), `host_neighbourhood`, `neighbourhood` (free-text version; we already use the cleaned `neighbourhood_cleansed`), `neighborhood_overview`, `host_about`, `host_location`, `license`. These aren't used anywhere in the project, so we leave them as-is rather than spending cleaning effort on columns we'll never read.
2. **Reserved for Part 2 (text modality)** — `description`. This will be processed separately (TF-IDF / embeddings), not filled with a placeholder here.
3. **Raw versions of already-cleaned columns** — `bathrooms`, `bathrooms_text` (superseded by `bathrooms_parsed`, which has zero missing values).
4. **Legitimately missing dates** — `host_since`, `first_review`, `last_review` are converted to real `datetime64`. The missing values here are real signal, not dirty data: a missing `first_review`/`last_review` simply means the listing has zero reviews. Each column has a companion `_was_missing` flag so this isn't silently lost.
5. **Negligible (<1% missing) ID/metadata/night-range columns** — `host_listings_count`, `host_verifications`, `host_has_profile_pic`, `minimum_minimum_nights`, etc. A handful of listings (2–11 rows) are missing these; not worth dropping rows or imputing for such a small count.

# 5. Saving The Processed Dataset

In [ ]:
# Save the cleaned dataset to data/processed/
output_path = PROCESSED_DIR / 'listings_cleaned.csv'
df_listings.to_csv(output_path, index=False)

print(f'Saved cleaned dataset to: {output_path}')
print(f'Shape: {df_listings.shape}')